# Práctica 04: Implementación de Limpieza de Datos - RENIEC
**Objetivo:** Aplicar forward-fill, casteo numérico, estandarización y validar antes/después.

## 1. Pipeline de limpieza

In [1]:
import glob, re, numpy as np, pandas as pd

def limpiar(f):
    with open(f, 'r', encoding='latin1') as fp:
        lines = [fp.readline() for _ in range(8)]
    sep = ';' if sum(l.count(';') for l in lines) > sum(l.count(',') for l in lines) else ','
    skip = next((i for i, l in enumerate(lines) if any(k in l.upper() for k in ['LOCAL', 'DEPARTAMENTO', 'TRANSAC'])), 0)
    df = pd.read_csv(f, sep=sep, encoding='latin1', skiprows=skip)
    df.columns = [c.strip().upper() for c in df.columns]
    cmap = {}
    for c in df.columns:
        if 'LOCAL' in c or 'CENTRO' in c: cmap[c] = 'LOCAL'
        elif 'DEP' in c: cmap[c] = 'DEPARTAMENTO'
        elif 'PROV' in c: cmap[c] = 'PROVINCIA'
        elif 'DIST' in c: cmap[c] = 'DISTRITO'
        elif 'MES' in c: cmap[c] = 'MES'
        elif 'TIPO' in c: cmap[c] = 'TIPO_TRANSACCION'
        elif 'TRANSAC' in c: cmap[c] = 'TRANSACCIONES'
    df = df.rename(columns=cmap)
    df['ANIO'] = int(re.search(r'202[0-9]', f).group(0)) if re.search(r'202[0-9]', f) else 2023
    for c in ['ANIO', 'LOCAL', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'MES', 'TIPO_TRANSACCION', 'TRANSACCIONES']:
        if c not in df.columns: df[c] = np.nan
    # Forward-fill de celdas combinadas de Excel
    for col in ['LOCAL', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'MES']:
        df[col] = df[col].ffill()
    return df[['ANIO', 'LOCAL', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'MES', 'TIPO_TRANSACCION', 'TRANSACCIONES']]

archivos = sorted(glob.glob("../Dataset-project/**/*.csv", recursive=True))
df_limpio = pd.concat([limpiar(f) for f in archivos], ignore_index=True)

# Casteo numérico y filtrado de subtotales
df_limpio['TRANSACCIONES'] = pd.to_numeric(df_limpio['TRANSACCIONES'].astype(str).str.replace(',', '').str.strip(), errors='coerce')
df_limpio = df_limpio.dropna(subset=['TRANSACCIONES', 'TIPO_TRANSACCION', 'LOCAL'])
df_limpio = df_limpio[df_limpio['TRANSACCIONES'] > 0]

# Normalización de texto
for col in ['LOCAL', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'MES', 'TIPO_TRANSACCION']:
    df_limpio[col] = df_limpio[col].astype(str).str.strip().str.upper().str.replace('¢', 'Ó').str.replace('Ã³', 'Ó')

df_limpio['DISTRITO'] = df_limpio['DISTRITO'].replace('NAN', 'NO REGISTRADO')
df_limpio['TRANSACCIONES'] = df_limpio['TRANSACCIONES'].astype(int)
print(f"Filas depuradas: {len(df_limpio):,}")

Filas depuradas: 88,584


## 2. Exportación de dataset limpio

In [2]:
df_limpio.to_csv("reniec_transacciones_limpio.csv", index=False)
print("Archivo generado: reniec_transacciones_limpio.csv")

Archivo generado: reniec_transacciones_limpio.csv


## 3. Resumen Antes vs Después

In [3]:
pd.DataFrame({
    "Indicador": ["Filas", "Nulos en LOCAL", "Nulos en DEPARTAMENTO", "Oficinas únicas", "Total Transacciones"],
    "Antes": ["122,527", "65,366 (53.3%)", "66,291 (54.1%)", "Inconsistente", "No evaluable"],
    "Después": [f"{len(df_limpio):,}", "0 (0.0%)", "0 (0.0%)", f"{df_limpio['LOCAL'].nunique():,}", f"{df_limpio['TRANSACCIONES'].sum():,}"]
})

,Indicador,Antes,Después
0,Filas,"122,527","88,584"
1,Nulos en LOCAL,"65,366 (53.3%)",0 (0.0%)
2,Nulos en DEPARTAMENTO,"66,291 (54.1%)",0 (0.0%)
3,Oficinas únicas,Inconsistente,"2,316"
4,Total Transacciones,No evaluable,"51,216,206"
